## ЗАДАНИЕ ДЛЯ ПРОГРАММИРУЮЩИХ

В этом задании мы будем использовать несколько библиотек для работы с текстом: [Natasha](https://github.com/natasha/natasha) для сегментации текста и приведения слов в предложениях к их нормальной форме и [NLTK](https://www.nltk.org/) для работы со стоп словами и вычисления статистических характеристик полученных результатов.

In [24]:
!pip install -q natasha

### Небольшой пример использования библиотеки Natasha

Рассмотрим пайплайн лемматизации текста с использованием Natasha

In [25]:
from natasha import Segmenter, NewsEmbedding, NewsMorphTagger, NewsSyntaxParser, Doc, MorphVocab
# Инициализируем компоненты Natasha
segmenter = Segmenter()
emb = NewsEmbedding()
morph_vocab = MorphVocab()
morph_tagger = NewsMorphTagger(emb)

Возьмем пару произвольныйх предложений в виде текста и для начала создадим объект `Doc`. Будем называть результат такого действия документом.

In [26]:
example_sentence = '''Это первое предложение. А это — второе предложение.'''

example_doc = Doc(example_sentence)
example_doc

Doc(text='Это первое предложение. А это — второе предложени...)

Видно, что полученный документ пока что содержит только атрибут `.text`. Произведем сегментацию, в результате которой у документа появятся дополнительные атрибуты: предложения (`.sents`) и токены (`.tokens`), причем внутри каждого предложения также содержатся токены, но только для этого предложения.

In [27]:
example_doc.segment(segmenter)
# Предложения
example_doc.sents

[DocSent(stop=23, text='Это первое предложение.', tokens=[...]),
 DocSent(start=24, stop=51, text='А это — второе предложение.', tokens=[...])]

In [28]:
# Токены самого первого предложения
example_doc.sents[0].tokens

[DocToken(stop=3, text='Это'),
 DocToken(start=4, stop=10, text='первое'),
 DocToken(start=11, stop=22, text='предложение'),
 DocToken(start=22, stop=23, text='.')]

In [29]:
# Все токены документа
example_doc.tokens

[DocToken(stop=3, text='Это'),
 DocToken(start=4, stop=10, text='первое'),
 DocToken(start=11, stop=22, text='предложение'),
 DocToken(start=22, stop=23, text='.'),
 DocToken(start=24, stop=25, text='А'),
 DocToken(start=26, stop=29, text='это'),
 DocToken(start=30, stop=31, text='—'),
 DocToken(start=32, stop=38, text='второе'),
 DocToken(start=39, stop=50, text='предложение'),
 DocToken(start=50, stop=51, text='.')]

При помощи `morph_tagger` в документ можно добавить морфологическую разметку. В результате у токенов добавятся атрибуты, отвечающие за часть речи, число и так далее.

In [30]:
# Морфологическая разметка
example_doc.tag_morph(morph_tagger)
example_doc.sents[0].tokens

[DocToken(stop=3, text='Это', pos='PRON', feats=<Inan,Nom,Neut,Sing>),
 DocToken(start=4, stop=10, text='первое', pos='ADJ', feats=<Nom,Pos,Neut,Sing>),
 DocToken(start=11, stop=22, text='предложение', pos='NOUN', feats=<Inan,Nom,Neut,Sing>),
 DocToken(start=22, stop=23, text='.', pos='PUNCT')]

После добавления морфологической разметки можно произвести лемматизацию токенов, то есть привести их к нормальной форме. В результате у токенов добавится атрибут `.lemma`

In [31]:
for token in example_doc.tokens:
  token.lemmatize(morph_vocab)

In [32]:
example_doc.tokens

[DocToken(stop=3, text='Это', pos='PRON', feats=<Inan,Nom,Neut,Sing>, lemma='это'),
 DocToken(start=4, stop=10, text='первое', pos='ADJ', feats=<Nom,Pos,Neut,Sing>, lemma='первый'),
 DocToken(start=11, stop=22, text='предложение', pos='NOUN', feats=<Inan,Nom,Neut,Sing>, lemma='предложение'),
 DocToken(start=22, stop=23, text='.', pos='PUNCT', lemma='.'),
 DocToken(start=24, stop=25, text='А', pos='CCONJ', lemma='а'),
 DocToken(start=26, stop=29, text='это', pos='PRON', feats=<Inan,Nom,Neut,Sing>, lemma='это'),
 DocToken(start=30, stop=31, text='—', pos='PUNCT', lemma='—'),
 DocToken(start=32, stop=38, text='второе', pos='ADJ', feats=<Nom,Pos,Neut,Sing>, lemma='второй'),
 DocToken(start=39, stop=50, text='предложение', pos='NOUN', feats=<Inan,Nom,Neut,Sing>, lemma='предложение'),
 DocToken(start=50, stop=51, text='.', pos='PUNCT', lemma='.')]

Можно теперь все лемматизированные токены собрать в один список

In [33]:
example_lemmas = [token.lemma for token in example_doc.tokens]
example_lemmas

['это',
 'первый',
 'предложение',
 '.',
 'а',
 'это',
 '—',
 'второй',
 'предложение',
 '.']

### Подготовка данных

Скачиваем текст, по которому будет дано задание, с помощью `urllib`

**Ссылка**, на источник текста

In [34]:
DATA_URL = "http://lib.ru/LITRA/LERMONTOW/geroi.txt"

In [35]:
import urllib.request

opener = urllib.request.URLopener({})
resource = opener.open(DATA_URL)
raw_text = resource.read().decode(resource.headers.get_content_charset()) #Текс с html тегами

In [36]:
raw_text[:200]

'<html><head><title>Михаил Лермонтов. Герой нашего времени</title></head><body><pre><div align=left><form action=/LITRA/LERMONTOW/geroi.txt><select name=format><OPTION VALUE=".fb2.zip">Fb2.zip<OPTION V'

Как видно, текст содержит html теги, от которых нужно избавиться. Выбрасываем из текста HTML-теги с помощью библиотеки Beatiful soap

In [37]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(raw_text, features="html.parser")

# удаляем в элементы скриптов и стилей
for script in soup(["script", "style"]):
    script.extract()

# забираем текст
cleaned_text = soup.get_text()

In [38]:
cleaned_text[:200]

'Михаил Лермонтов. Герой нашего времениFb2.zipEpubСодержаниеFine HTMLPrinted versiontxt(Word,КПК)Lib.ru html\nМихаил Лермонтов. Герой нашего времени\n\n\n     Во всякой книге предисловие есть первая и вмес'

Некоторые лишние символы все равно остались, например, те же `\r` и `\n`, и можно было углубиться в процедуру очистки, однако, в данной работе это не является самоцелью, поэтому будем довольствоваться полученным результатом.

### Задание 1

Используя очищенный текст (переменная `cleaned_text`) и рассмотренный пример лемматизации с помощью библиотеки Natasha, произведите лемматизацию очищенного текста. С помощью метода `str.isalpha` из стандартной библиотеки Python модифицируйте рассмотренный в примере код так, чтобы в результате остались **только буквенные токены**. Иными словами, результат — список лемм токенов очищенного текста, которые состоят строго из букв.

In [39]:

doc = Doc(cleaned_text)
doc.segment(segmenter)
doc.tag_morph(morph_tagger)

non_uniq_tokens = []
for token in doc.tokens:
    token.lemmatize(morph_vocab)
    if token.lemma.isalpha():
        non_uniq_tokens.append(token.lemma)


In [40]:
non_uniq_tokens[:10]

['михаил',
 'лермонтов',
 'герой',
 'наш',
 'время',
 'fb',
 'zipepub',
 'содержание',
 'fine',
 'htmlprinted']

In [41]:
doc = Doc(cleaned_text)
doc.segment(segmenter)
doc.tag_morph(morph_tagger)

non_uniq_tokens = []
for token in doc.tokens:
    token.lemmatize(morph_vocab)
    if token.lemma and token.lemma.isalpha():
        non_uniq_tokens.append(token.lemma.lower())

len(doc.sents), len(non_uniq_tokens)

(2970, 41693)

Проверьте себя. Должны получиться следующие значения:

*   Предложений: 559 (возможны расхождения в несколько предложений)
*   Токенов: примерно 8640 (возможны расхождения в некоторое количество токенов)

In [42]:
len(doc.sents)

2970

In [43]:
len(non_uniq_tokens)

41693

Для продолжения работы над заданием числа должны быть близки к указанным

## Задание 2

Используя `non_uniq_tokens`, стоп-слова для русского языка из библиотеки NLTK (`nltk.corpus.stopwords`) и `nltk.FreqDist`, вычислите, **какую долю среди 100 самых частотных** токенов в произведении занимают токены, **не относящиеся** к стоп словам.

**Например**, если среди 100 самых частотных слов встречается 25 слов, входящих в стоп лист, значит не входят в стоп лист 75 слов, и их доля составит 0.75.

**Не бойтесь использовать документацию NLTK и тьюториалы.**

In [44]:
import nltk
from nltk import FreqDist
from nltk.corpus import stopwords
nltk.download("stopwords")
STOPWORDS = set(stopwords.words("russian"))
stopwords.words("russian")[:5] #Пример стоп слов

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


['и', 'в', 'во', 'не', 'что']

In [49]:
freq_dist = FreqDist(non_uniq_tokens)
top_100 = [token for token, count in freq_dist.most_common(40)]
share_non_stop = round(sum(token not in STOPWORDS for token in top_100) / 40, 2)
share_non_stop


0.23

Проверьте себя: должно получиться 0.44 (допустимо небольшое расхождение)

## Задание 3
Вычислите, сколько токенов встречается в тексте **строго больше** 50 раз.

In [48]:

count_gt_50 = sum(count > 20 for count in freq_dist.values())
count_gt_50

256

Проверьте себя: должно получиться значение 24 (возможно небольшое расхождение)
